In [ ]:
!pip install gymnasium[atari,accept-rom-license] stable-baselines3 shimmy
!pip install opencv-python

In [ ]:
import os
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

# 1. SETUP ENVIRONMENT
def make_env():
    # 'full_action_space=False' focuses on [NOOP, UP, DOWN]
    env = gym.make("ALE/Freeway-v5", render_mode="rgb_array", full_action_space=False)
    # Preprocessing (Level 4): MaxAndSkip, Grayscale, Resize 84x84
    env = AtariWrapper(env) 
    return env

# Vectorize
env = DummyVecEnv([make_env])

# 2. DIRECTORY FOR SAVES
save_path = "./freeway_model_1/"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 3. MODEL 1 CONFIGURATION (Simplified)
model_1 = DQN(
    "CnnPolicy", 
    env, 
    verbose=0,                 # Keep 0 for the progress bar
    buffer_size=100000,        # RUBRIC REQ: 100k data points
    learning_rate=1e-4,
    batch_size=32,
    device="auto"              # Auto-detects your GTX 1660
)

# 4. CHECKPOINT CALLBACK
checkpoint_callback = CheckpointCallback(
    save_freq=10000, 
    save_path=save_path, 
    name_prefix="model1_baseline"
)

# 5. START TRAINING
print("🚀 Training Model 1... (Live Progress Bar Enabled)")
try:
    model_1.learn(
        total_timesteps=200000, 
        callback=checkpoint_callback,
        progress_bar=True      # Live visual update
    )
    
    # 6. SAVE FINAL MODEL
    model_1.save(os.path.join(save_path, "model_1_final"))
    print(f"\n✅ Success! Model 1 saved in {save_path}")
    
except KeyboardInterrupt:
    print("\n⏹️ Training paused. Progress saved.")
except Exception as e:
    print(f"❌ An error occurred: {e}")